In [1]:

# 1. Импорт библиотек

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold



# 2. Загрузка данных

sheet_id = "1q-nbWuFrfrIBMXmZfNW78N3bx5v60Vb9"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(url)



# 3. Очистка данных

df = df.drop(columns=['Unnamed: 0'])
df = df.fillna(df.median(numeric_only=True))

selector = VarianceThreshold(threshold=0.01)
X_temp = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])

selector.fit(X_temp)

low_var_cols = X_temp.columns[~selector.get_support()]
df = df.drop(columns=low_var_cols)



# 4. Подготовка таргета

ic50_median = df['IC50, mM'].median()
df['IC50_class'] = (df['IC50, mM'] > ic50_median).astype(int)

X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI', 'IC50_class'])
y = df['IC50_class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



# 5. Logistic Regression

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

y_pred_lr = logreg.predict(X_test)

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))



# 6. Random Forest

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\nRandom Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

/tmp/ipykernel_7134/792713990.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['IC50_class'] = (df['IC50, mM'] > ic50_median).astype(int)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", l

Logistic Regression
Accuracy: 0.5174129353233831
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        97
           1       0.52      1.00      0.68       104

    accuracy                           0.52       201
   macro avg       0.26      0.50      0.34       201
weighted avg       0.27      0.52      0.35       201


Random Forest
Accuracy: 0.7412935323383084
              precision    recall  f1-score   support

           0       0.71      0.77      0.74        97
           1       0.77      0.71      0.74       104

    accuracy                           0.74       201
   macro avg       0.74      0.74      0.74       201
weighted avg       0.74      0.74      0.74       201



### Что получилось

**Logistic Regression**
- Accuracy: **0.517**
- Модель предсказывает почти только класс **1**
- Класс 0 не определяется:
   precision = 0.00
   recall = 0.00

**Random Forest**
- Accuracy: **0.741**
- Класс 0:
  precision: 0.71
 recall: 0.77
- Класс 1:
   precision: 0.77
   recall: 0.71

### Вывод

- Logistic Regression работает плохо и даёт перекос в один класс  
- Random Forest показывает заметно более устойчивый результат  
- Классы разделяются умеренно хорошо, но не так сильно, как в предыдущих черновых запусках  

 для задачи **IC50 > медианы** оставляю **Random Forest** как лучшую модель

### Финальный вывод по классификации IC50

- Лучшая модель: **Random Forest**
- Accuracy: **≈ 0.74**
- Модель даёт сбалансированные результаты по обоим классам  

- Logistic Regression не справилась (предсказывает один класс)  

 классификация работает лучше, чем регрессия IC50  

Следующее:
- создаю файл **classification_cc50.ipynb**